[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn as nn
import math

In [5]:
help(torch.repeat_interleave)

Help on built-in function repeat_interleave in module torch:

repeat_interleave(...)
    repeat_interleave(input, repeats, dim=None, *, output_size=None) -> Tensor

    Repeat elements of a tensor.

    .. warning::

        This is different from :meth:`torch.Tensor.repeat` but similar to ``numpy.repeat``.

    Args:
        input (Tensor): the input tensor.
        repeats (Tensor or int): The number of repetitions for each element.
            repeats is broadcasted to fit the shape of the given axis.
        dim (int, optional): The dimension along which to repeat values.
            By default, use the flattened input array, and return a flat output
            array.

    Keyword args:
        output_size (int, optional): Total output size for the given axis
            ( e.g. sum of repeats). If given, it will avoid stream synchronization
            needed to calculate output shape of the tensor.

    Returns:
        Tensor: Repeated tensor which has the same shape as input, e

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = self.d_model // self.num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, seq, _ = x.shape
        q = self.W_q(x).view(B, seq, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, seq, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, seq, self.num_kv_heads, self.d_k).transpose(1, 2)
        
        c = self.num_heads // self.num_kv_heads
        k = torch.repeat_interleave(k, c, dim=1)
        v = torch.repeat_interleave(v, c, dim=1)

        score = q @ k.transpose(-1, -2) / math.sqrt(self.d_k)
        
        weight = torch.softmax(score, dim=-1)
        out = weight @ v
        return self.W_o(out.transpose(1, 2).reshape(B, seq, -1))

In [7]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [8]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.6ms)
  ✅ [2/5] nn.Linear with correct shapes (1.1ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.1ms)
  ✅ [4/5] KV heads are shared correctly (4.8ms)
  ✅ [5/5] Gradient flow (31.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (44.2ms total)
  Progress saved. Run status() to see your dashboard.

